# CornerScout - 04 Ingenieria de variables prepartido

**Responsabilidad:** consumir exclusivamente `03-scr15-v2`, auditar el proxy de
corner corto, construir historicos de ocho partidos estrictamente anteriores y
exportar candidatos prepartido trazables.

Este notebook no reconstruye SCR-15 y no usa informacion del corner objetivo
para sus predictores. K-Means se conserva como descripcion de destinos, nunca
como predictor ni como evidencia de una jugada ensayada.

In [ ]:
import json
import io
import math
import platform
import zipfile
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.metrics import (adjusted_rand_score, balanced_accuracy_score,
                              cohen_kappa_score, confusion_matrix,
                              matthews_corrcoef, precision_recall_fscore_support,
                              silhouette_score)

from analytics.io import data_dir, digest, write_json

DATA = data_dir()
INPUT = DATA / "interim" / "03_scr15"
MATCH_INPUT = DATA / "interim" / "02_clean" / "matches_clean.parquet"
LABEL_INPUT = DATA / "manual_labels" / "short_corner_review.csv"
OUT = DATA / "processed" / "04_features"
OUT.mkdir(parents=True, exist_ok=True)

STAGE_VERSION = "04-features-v2"
SOURCE_CONTRACT = "03-scr15-v2"
RULE_VERSION = "scr15-research-v1.2-first-limit"
HISTORY_MATCHES = 8
SHORT_THRESHOLD = 18.0
DEVELOPMENT_CUTOFF = pd.Timestamp("2016-01-01")
EDA_END = DEVELOPMENT_CUTOFF
SMOOTHING_STRENGTH = 10.0
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")

CORE_FEATURES = [
    "hist_corners_per_match", "hist_scr15_smoothed", "hist_xg_per_corner",
    "hist_short_share", "hist_high_share", "hist_top_taker_share",
    "hist_dominant_zone_share", "hist_score_losing_share",
    "opp_hist_scr15_conceded_smoothed", "is_home",
]
print({"run_id": RUN_ID, "stage": STAGE_VERSION, "python": platform.python_version(),
       "pandas": pd.__version__, "sklearn": sklearn.__version__})

## 1. Contrato de entrada

La etapa se detiene si version, hash o conteos no corresponden a la ejecucion
auditada de SCR-15. Las seis secuencias desconocidas permanecen nulas.

In [ ]:
contract03 = json.loads((INPUT / "contract.json").read_text(encoding="utf-8"))
corner_path = INPUT / "corners_scr15.parquet"
corner_export = next(item for item in contract03["exports"] if item["file"] == corner_path.name)
assert contract03["stage"] == "03_scr15"
assert contract03["contract_version"] == SOURCE_CONTRACT
assert contract03["rule_version"] == RULE_VERSION
assert contract03["source_contract_version"] == "02-clean-v2"
assert digest(corner_path) == corner_export["sha256"]
assert contract03["counts"] == {
    "matches": 380, "events": 1295354, "corners": 3841, "evaluable": 3835,
    "excluded": 6, "with_shot": 1245, "shared_shots": 0,
    "new_corner_closures": 29, "restarts_audited": 105,
}

corners = pd.read_parquet(corner_path)
matches = pd.read_parquet(MATCH_INPUT)
labels = pd.read_csv(LABEL_INPUT, dtype={"event_id": "string"})
corners["match_date"] = pd.to_datetime(corners["match_date"]).dt.normalize()
matches["match_date"] = pd.to_datetime(matches["match_date"]).dt.normalize()

CORNER_COLUMNS = {
    "event_id", "match_id", "match_date", "team", "opponent", "rule_version",
    "valid_sequence", "shot_within_15s", "xg_sequence", "x", "y", "end_x", "end_y",
    "height", "player", "game_state", "match_minute", "score_diff", "player_difference",
    "match_phase", "numerical_state", "repeat_corner_60s",
    "seconds_since_previous_same_team_corner", "attacking_players", "defending_players",
}
MATCH_COLUMNS = {"match_id", "match_date", "kick_off", "home_team", "away_team"}
assert not (CORNER_COLUMNS - set(corners.columns)), sorted(CORNER_COLUMNS - set(corners.columns))
assert not (MATCH_COLUMNS - set(matches.columns)), sorted(MATCH_COLUMNS - set(matches.columns))

assert len(corners) == 3841 and corners.event_id.is_unique
assert corners.rule_version.eq(RULE_VERSION).all()
assert int(corners.valid_sequence.sum()) == 3835
assert corners.loc[corners.valid_sequence, "shot_within_15s"].isin([0, 1]).all()
assert corners.loc[~corners.valid_sequence, "shot_within_15s"].isna().all()
assert int(corners.loc[corners.valid_sequence, "shot_within_15s"].sum()) == 1245
assert len(matches) == 380 and matches.match_id.is_unique
print(f"Entrada aprobada: {len(matches)} partidos, {len(corners):,} corners, "
      f"{corners.valid_sequence.sum():,} evaluables")

## 2. Variables tacticas observadas

El proxy corto requiere geometria valida, pase `Ground Pass` o `Low Pass` y
distancia no mayor de 18 unidades StatsBomb. Es una regla descriptiva versionada,
no una etiqueta humana validada.

In [ ]:
corners = corners.copy()
corners["valid_geometry"] = (
    corners[["x", "y", "end_x", "end_y"]].notna().all(axis=1)
    & corners.x.between(0, 120) & corners.y.between(0, 80)
    & corners.end_x.between(0, 120) & corners.end_y.between(0, 80)
    & corners.x.ge(118) & (corners.y.le(2) | corners.y.ge(78))
)
corners["pass_length"] = np.hypot(corners.end_x - corners.x, corners.end_y - corners.y)
corners["end_y_relative"] = np.where(corners.y < 40, corners.end_y, 80 - corners.end_y)
corners["short_proxy"] = (
    corners.valid_geometry & corners.height.isin(["Ground Pass", "Low Pass"])
    & corners.pass_length.le(SHORT_THRESHOLD)
)
corners["execution_type"] = np.select(
    [corners.short_proxy, corners.valid_geometry], ["short", "direct"], default="unknown"
)
corners["direct_delivery_valid"] = corners.valid_geometry & ~corners.short_proxy
corners["delivery_zone"] = pd.Series(pd.NA, index=corners.index, dtype="string")
outside = corners.end_x.lt(102) | corners.end_y_relative.lt(18) | corners.end_y_relative.gt(62)
corners.loc[corners.direct_delivery_valid & outside, "delivery_zone"] = "fuera_area"
corners.loc[corners.direct_delivery_valid & ~outside & corners.end_y_relative.lt(36), "delivery_zone"] = "franja_cercana"
corners.loc[corners.direct_delivery_valid & ~outside & corners.end_y_relative.between(36, 44), "delivery_zone"] = "franja_central"
corners.loc[corners.direct_delivery_valid & ~outside & corners.end_y_relative.gt(44), "delivery_zone"] = "franja_lejana"
corners.loc[~corners.direct_delivery_valid, "delivery_zone"] = pd.NA
corners["corner_side"] = np.where(corners.y < 40, "left", "right")
corners["is_high"] = corners.height.eq("High Pass")

assert int(corners.valid_geometry.sum()) == 3838
assert int(corners.short_proxy.sum()) == 464
assert int(corners.direct_delivery_valid.sum()) == 3374
assert corners.loc[corners.direct_delivery_valid, "delivery_zone"].notna().all()
display(corners[["valid_geometry", "short_proxy", "direct_delivery_valid"]].sum().to_frame("n"))

## 3. Auditoria de las 40 etiquetas heredadas

Se auditan integridad, enlace, diseno muestral y acuerdo. Son 40 casos heredados
para calibracion descriptiva del proxy, no una validacion independiente: no hay
evidencia de cegamiento o doble anotacion y el mismo conjunto fue usado para
explorar umbrales. Ningun umbral se elige por su relacion con SCR-15.

In [ ]:
assert len(labels) == 40 and labels.event_id.notna().all() and labels.event_id.is_unique
assert set(labels.columns) == {"event_id", "manual_short_label", "manual_notes"}
assert labels.manual_short_label.isin([0, 1]).all() and labels.manual_notes.notna().all()
review = corners.merge(labels, on="event_id", how="inner", validate="one_to_one")
assert len(review) == 40 and set(review.event_id) == set(labels.event_id)
assert review.valid_geometry.all() and review.valid_sequence.all()

legacy_zip = DATA / "processed" / "04_features-20260919T170245Z-1-001.zip"
with zipfile.ZipFile(legacy_zip) as archive:
    legacy_review = pd.read_parquet(io.BytesIO(archive.read("04_features/short_manual_review.parquet")))
compare_columns = ["event_id", "manual_short_label", "manual_notes"]
pd.testing.assert_frame_equal(
    labels[compare_columns].sort_values("event_id").reset_index(drop=True),
    legacy_review[compare_columns].astype({"event_id": "string"}).sort_values("event_id").reset_index(drop=True),
    check_dtype=False,
)

review["distance_band"] = pd.cut(
    review.pass_length, [-np.inf, 12, 15, 18, np.inf],
    labels=["<=12", "(12,15]", "(15,18]", ">18"],
)
y_true = review.manual_short_label.astype(int)
y_pred = review.short_proxy.astype(int)
tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="binary", zero_division=0
)
label_metrics = pd.DataFrame([{
    "n": len(review), "positive": int(y_true.sum()), "negative": int((1-y_true).sum()),
    "accuracy": float((y_true == y_pred).mean()), "precision": precision,
    "recall": recall, "specificity": tn / (tn + fp), "f1": f1,
    "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    "cohen_kappa": cohen_kappa_score(y_true, y_pred),
    "mcc": matthews_corrcoef(y_true, y_pred), "tp": int(tp), "tn": int(tn),
    "fp": int(fp), "fn": int(fn),
}])
review["note_contradiction"] = review.manual_short_label.eq(0) & review.manual_notes.str.contains(
    "corto|cercano", case=False, regex=True
)
height_identity = review.manual_short_label.eq(review.height.eq("Ground Pass").astype(int)).all()
label_audit = {
    "rows": 40, "unique_ids": 40, "joined": 40,
    "positive": int(y_true.sum()), "negative": int((1-y_true).sum()),
    "contradictory_negative_notes": int(review.note_contradiction.sum()),
    "labels_equal_ground_pass_rule": bool(height_identity),
    "sample_design": "10 cases from each of four distance bands; not prevalence-representative",
    "methodology_independent": False, "adjudicated": False,
    "status": "legacy_labels_integrity_verified_methodology_not_validated",
    "matches_legacy_short_manual_review": True,
}
assert label_audit["contradictory_negative_notes"] == 10 and height_identity
display(label_metrics.round(4))
display(pd.crosstab(review.distance_band, review.manual_short_label, margins=True))
review["distance_from_threshold"] = review.pass_length - SHORT_THRESHOLD
near_threshold = review[review.distance_from_threshold.abs().le(2)][
    ["event_id", "manual_short_label", "pass_length", "distance_from_threshold", "height", "short_proxy"]
].sort_values("pass_length")
threshold_rows = []
for threshold in [15.0, 18.0, 20.0]:
    season_prediction = corners.valid_geometry & corners.height.isin(["Ground Pass", "Low Pass"]) & corners.pass_length.le(threshold)
    review_prediction = review.valid_geometry & review.height.isin(["Ground Pass", "Low Pass"]) & review.pass_length.le(threshold)
    threshold_rows.append({
        "threshold": threshold, "season_short_count": int(season_prediction.sum()),
        "review_short_count": int(review_prediction.sum()),
        "review_agreement": float(review.manual_short_label.eq(review_prediction.astype(int)).mean()),
    })
threshold_sensitivity = pd.DataFrame(threshold_rows)
assert threshold_sensitivity.loc[threshold_sensitivity.threshold.eq(18), "season_short_count"].iloc[0] == 464
display(review[["event_id", "manual_short_label", "pass_length", "distance_from_threshold"]].sort_values("pass_length"))
display(near_threshold)
display(threshold_sensitivity)
print(json.dumps(label_audit, ensure_ascii=False, indent=2))

## 4. Unidad equipo-partido e historicos

Cada partido genera dos perspectivas, incluso con cero corners. Para cada
objetivo se seleccionan exactamente los ocho partidos del equipo con
`match_date < target_date`; el mismo criterio se aplica a la defensa rival.

In [ ]:
def observed_row(match, team):
    opponent = match.away_team if team == match.home_team else match.home_team
    group = corners[(corners.match_id == match.match_id) & (corners.team == team)]
    evaluable = group[group.valid_sequence]
    return {
        "match_id": int(match.match_id), "match_date": match.match_date,
        "kick_off": match.kick_off, "team": team, "opponent": opponent,
        "is_home": int(team == match.home_team), "n_corners": len(group),
        "n_evaluable": len(evaluable),
        "n_with_shot": int(evaluable.shot_within_15s.sum()),
        "xg_sum": float(evaluable.xg_sequence.fillna(0).sum()),
        "n_short": int(group.short_proxy.sum()), "n_high": int(group.is_high.sum()),
        "n_losing": int(group.game_state.eq("losing").sum()),
    }

observed = pd.DataFrame(
    observed_row(match, team)
    for match in matches.sort_values(["match_date", "kick_off", "match_id"]).itertuples()
    for team in (match.home_team, match.away_team)
)
assert len(observed) == 760 and observed.groupby("match_id").size().eq(2).all()
assert int(observed.n_corners.sum()) == 3841
assert int(observed.n_evaluable.sum()) == 3835
assert int(observed.n_corners.eq(0).sum()) == 17

def prior_rows(team, cutoff):
    return observed[(observed.team == team) & (observed.match_date < cutoff)].sort_values(
        ["match_date", "kick_off", "match_id"]
    ).tail(HISTORY_MATCHES)

def ratio(numerator, denominator):
    return float(numerator / denominator) if denominator else np.nan

def historical_features(row, source_corners=corners):
    history = prior_rows(row.team, row.match_date)
    opponent_history = prior_rows(row.opponent, row.match_date)
    result = {
        "match_id": row.match_id, "match_date": row.match_date, "team": row.team,
        "opponent": row.opponent, "is_home": row.is_home,
        "history_n_matches": len(history), "opp_history_n_matches": len(opponent_history),
        "history_match_ids": json.dumps(history.match_id.astype(int).tolist()),
        "opp_history_match_ids": json.dumps(opponent_history.match_id.astype(int).tolist()),
        "history_max_date": history.match_date.max() if len(history) else pd.NaT,
        "opp_history_max_date": opponent_history.match_date.max() if len(opponent_history) else pd.NaT,
    }
    ready = len(history) == HISTORY_MATCHES and len(opponent_history) == HISTORY_MATCHES
    result["history_ready"] = len(history) == HISTORY_MATCHES
    result["pre_match_ready"] = ready
    if not ready:
        return result

    history_ids = set(history.match_id.astype(int))
    team_corners = source_corners[source_corners.match_id.isin(history_ids) & (source_corners.team == row.team)]
    team_eval = team_corners[team_corners.valid_sequence]
    league_prior = source_corners[(source_corners.match_date < row.match_date) & source_corners.valid_sequence]
    league_rate = ratio(league_prior.shot_within_15s.sum(), len(league_prior))
    team_shots, team_n = team_eval.shot_within_15s.sum(), len(team_eval)

    opp_ids = set(opponent_history.match_id.astype(int))
    conceded = source_corners[source_corners.match_id.isin(opp_ids) & (source_corners.team != row.opponent)]
    conceded_eval = conceded[conceded.valid_sequence]
    conceded_shots, conceded_n = conceded_eval.shot_within_15s.sum(), len(conceded_eval)
    taker_counts = team_corners.player.value_counts()
    zone_counts = team_corners.delivery_zone.value_counts()
    result.update({
        "hist_corners_per_match": len(team_corners) / HISTORY_MATCHES,
        "hist_scr15_raw": ratio(team_shots, team_n),
        "hist_scr15_smoothed": (team_shots + SMOOTHING_STRENGTH * league_rate) / (team_n + SMOOTHING_STRENGTH),
        "hist_xg_per_corner": ratio(team_eval.xg_sequence.fillna(0).sum(), team_n),
        "hist_short_share": ratio(team_corners.short_proxy.sum(), len(team_corners)),
        "hist_high_share": ratio(team_corners.is_high.sum(), len(team_corners)),
        "hist_top_taker_share": ratio(taker_counts.max() if len(taker_counts) else 0, len(team_corners)),
        "hist_dominant_zone_share": ratio(zone_counts.max() if len(zone_counts) else 0, len(team_corners)),
        "hist_score_losing_share": ratio(team_corners.game_state.eq("losing").sum(), len(team_corners)),
        "opp_hist_scr15_conceded_raw": ratio(conceded_shots, conceded_n),
        "opp_hist_scr15_conceded_smoothed": (conceded_shots + SMOOTHING_STRENGTH * league_rate) / (conceded_n + SMOOTHING_STRENGTH),
        "league_prior_scr15": league_rate,
    })
    return result

features = pd.DataFrame(historical_features(row) for row in observed.itertuples())
assert len(features) == 760 and int(features.pre_match_ready.sum()) == 600
ready = features[features.pre_match_ready]
assert ready.history_n_matches.eq(8).all() and ready.opp_history_n_matches.eq(8).all()
assert (ready.history_max_date < ready.match_date).all()
assert (ready.opp_history_max_date < ready.match_date).all()
for row in ready.itertuples():
    assert row.match_id not in json.loads(row.history_match_ids)
    assert row.match_id not in json.loads(row.opp_history_match_ids)
    expected = prior_rows(row.team, row.match_date).match_id.astype(int).tolist()
    assert json.loads(row.history_match_ids) == expected

# Perturbar todos los resultados presentes y futuros no puede alterar el pasado del objetivo.
probe = ready.sort_values(["match_date", "match_id", "team"]).iloc[len(ready) // 2]
probe_observed = observed[(observed.match_id == probe.match_id) & (observed.team == probe.team)].iloc[0]
baseline_probe = historical_features(probe_observed)
perturbed = corners.copy()
future = perturbed.match_date.ge(probe.match_date) & perturbed.valid_sequence
perturbed.loc[future, "shot_within_15s"] = 1 - perturbed.loc[future, "shot_within_15s"].astype(int)
perturbed.loc[future, "xg_sequence"] = 999.0
mutated_probe = historical_features(probe_observed, perturbed)
PERTURBATION_FIELDS = [
    "hist_scr15_raw", "hist_scr15_smoothed", "hist_xg_per_corner",
    "opp_hist_scr15_conceded_raw", "opp_hist_scr15_conceded_smoothed", "league_prior_scr15",
]
for field in PERTURBATION_FIELDS:
    assert np.isclose(baseline_probe[field], mutated_probe[field], equal_nan=True), field
perturbation_audit = pd.DataFrame([{
    "target_match_id": int(probe.match_id), "target_date": probe.match_date,
    "mutated_rows_on_or_after_target": int(future.sum()), "fields_checked": json.dumps(PERTURBATION_FIELDS),
    "passed": True,
}])
display(features.pre_match_ready.value_counts().rename_axis("pre_match_ready").to_frame("team_matches"))

In [ ]:
candidate_base = corners[corners.valid_sequence].merge(
    features[features.pre_match_ready], on=["match_id", "match_date", "team", "opponent"],
    how="inner", validate="many_to_one", suffixes=("", "_history")
)
PRE_COLUMNS = [
    "event_id", "match_id", "match_date", "team", "opponent", "shot_within_15s",
    "history_match_ids", "opp_history_match_ids", "history_max_date", "opp_history_max_date",
    "history_n_matches", "opp_history_n_matches", "pre_match_ready", *CORE_FEATURES,
]
SCENARIO_COLUMNS = [
    "match_minute", "score_diff", "player_difference", "match_phase", "game_state",
    "numerical_state", "corner_side", "repeat_corner_60s",
    "seconds_since_previous_same_team_corner", "attacking_players", "defending_players",
]
training_pre = candidate_base[PRE_COLUMNS].copy()
training_scenario = candidate_base[PRE_COLUMNS + SCENARIO_COLUMNS].copy()
training_pre["shot_within_15s"] = training_pre.shot_within_15s.astype(int)
training_scenario["shot_within_15s"] = training_scenario.shot_within_15s.astype(int)
assert len(training_pre) == 3039 and training_pre.event_id.is_unique
assert training_pre.match_id.nunique() == 300 and training_pre.team.nunique() == 20
assert int(training_pre.shot_within_15s.sum()) == 976
assert set(training_pre.event_id) == set(training_scenario.event_id)
assert np.isfinite(training_pre[CORE_FEATURES].to_numpy(dtype=float)).all()
assert training_pre.groupby(["team", "match_id"])[CORE_FEATURES].nunique(dropna=False).le(1).all().all()

reconciled = {
    "0a7a6700-a960-4dbb-976d-d35ee1856c84", "ac90224b-64e7-43d7-9ec3-a325d88ec7f9",
    "7f36528e-a33f-4f3d-8092-c4f481079196", "4a33d3d6-7b09-4586-af70-d3a77ad17b94",
    "4240887d-22b8-44df-8f9a-5ef3cd02dce5",
}
reconciled_rows = training_pre[training_pre.event_id.isin(reconciled)]
assert set(reconciled_rows.event_id) == reconciled and reconciled_rows.shot_within_15s.eq(0).all()
print(f"Candidatos prepartido: {len(training_pre):,}; positivos: {training_pre.shot_within_15s.sum():,}")

# Cuatro poblaciones documentadas para modelacion posterior; aqui no se entrena ningun modelo.
all_corner_layers = corners.merge(
    features, on=["match_id", "match_date", "team", "opponent"], how="left",
    validate="many_to_one", suffixes=("", "_history")
)
KNOWN_BEFORE_KICK = [
    "match_minute", "score_diff", "player_difference", "match_phase", "game_state",
    "numerical_state", "corner_side", "repeat_corner_60s",
    "seconds_since_previous_same_team_corner", "attacking_players", "defending_players",
]
MODEL_TRACE = ["event_id", "match_id", "match_date", "team", "opponent", "pre_match_ready"]
MODEL_PREDICTORS = CORE_FEATURES + KNOWN_BEFORE_KICK

model_short_direct = all_corner_layers.loc[
    all_corner_layers.valid_geometry,
    MODEL_TRACE + MODEL_PREDICTORS + ["short_proxy"],
].copy()
model_short_direct["short_proxy"] = model_short_direct.short_proxy.astype(int)

model_delivery_zone = all_corner_layers.loc[
    all_corner_layers.direct_delivery_valid,
    MODEL_TRACE + MODEL_PREDICTORS + ["delivery_zone"],
].copy()
assert model_delivery_zone.delivery_zone.notna().all()

model_corner_count = observed.merge(
    features, on=["match_id", "match_date", "team", "opponent", "is_home"],
    how="left", validate="one_to_one", suffixes=("", "_history")
)
model_corner_count["exposure_matches"] = 1
COUNT_COLUMNS = [
    "match_id", "match_date", "team", "opponent", "pre_match_ready",
    "history_match_ids", "opp_history_match_ids", "n_corners", "exposure_matches", *CORE_FEATURES,
]
model_corner_count = model_corner_count[COUNT_COLUMNS]

assert len(model_short_direct) == 3838 and model_short_direct.event_id.is_unique
assert len(model_delivery_zone) == 3374 and model_delivery_zone.event_id.is_unique
assert len(model_corner_count) == 760 and model_corner_count.groupby("match_id").size().eq(2).all()
assert int(model_corner_count.n_corners.eq(0).sum()) == 17
assert model_corner_count.exposure_matches.eq(1).all()
for prohibited in ["height", "end_x", "end_y", "pass_length", "xg_sequence", "shot_within_15s"]:
    assert prohibited not in model_short_direct.columns and prohibited not in model_delivery_zone.columns

modeling_table_contract = pd.DataFrame([
    {"table": "model_scr15_pre_match", "unit": "evaluable corner with both histories ready",
     "target": "shot_within_15s", "predictor_layer": "pre_match", "rows": len(training_pre)},
    {"table": "model_scr15_scenario", "unit": "evaluable corner with both histories ready",
     "target": "shot_within_15s", "predictor_layer": "pre_match + known_before_kick", "rows": len(training_scenario)},
    {"table": "model_short_direct", "unit": "corner with valid geometry",
     "target": "short_proxy", "predictor_layer": "pre_match + known_before_kick", "rows": len(model_short_direct)},
    {"table": "model_delivery_zone", "unit": "valid direct delivery",
     "target": "delivery_zone (unchanged taxonomy)", "predictor_layer": "pre_match + known_before_kick", "rows": len(model_delivery_zone)},
    {"table": "model_corner_count", "unit": "team-match including zero-corner matches",
     "target": "n_corners", "predictor_layer": "pre_match; exposure_matches=1", "rows": len(model_corner_count)},
])
display(modeling_table_contract)

## 5. K-Means descriptivo

Se ajusta sobre destinos directos anteriores a 2016-01-01. El numero de grupos
es una particion descriptiva y no entra a los modelos supervisados. Los grupos
son destinos de pase en coordenadas alineadas, no posiciones de jugadores ni
evidencia de jugadas ensayadas. `k=4` se conserva como hipotesis visual, no como
resultado de optimizacion.

In [ ]:
cluster_pool = corners[
    corners.direct_delivery_valid & (corners.match_date < DEVELOPMENT_CUTOFF)
].copy()
cluster_x = cluster_pool[["end_x", "end_y_relative"]].to_numpy()
kmeans = KMeans(n_clusters=4, n_init=20, random_state=42).fit(cluster_x)
cluster_pool["cluster_id"] = kmeans.labels_
cluster_pool["distance_to_center"] = np.linalg.norm(cluster_x - kmeans.cluster_centers_[kmeans.labels_], axis=1)
cluster_centers = pd.DataFrame(kmeans.cluster_centers_, columns=["end_x", "end_y_relative"])
cluster_centers.insert(0, "cluster_id", range(4))
cluster_centers["y_order"] = cluster_centers.end_y_relative.rank(method="first").astype(int)
cluster_centers["geometric_name"] = cluster_centers.apply(
    lambda row: f"destino_y{int(row.y_order):02d}_x{int(round(row.end_x)):03d}", axis=1
)
cluster_centers["n"] = cluster_centers.cluster_id.map(cluster_pool.cluster_id.value_counts())
cluster_centers["fit_before"] = DEVELOPMENT_CUTOFF
cluster_centers["coordinate_transform"] = "end_x unchanged; end_y reflected as 80-end_y for high-side corners"
assert cluster_centers.geometric_name.is_unique
cluster_pool = cluster_pool.merge(
    cluster_centers[["cluster_id", "geometric_name"]], on="cluster_id", how="left", validate="many_to_one"
)
cluster_assignments = cluster_pool[
    ["event_id", "match_id", "match_date", "team", "cluster_id", "geometric_name", "distance_to_center"]
]

sensitivity_rows = []
seeds = [7, 21, 42, 84, 168]
for k in range(2, 9):
    reference = KMeans(n_clusters=k, n_init=20, random_state=42).fit(cluster_x)
    seed_aris = [
        adjusted_rand_score(reference.labels_, KMeans(n_clusters=k, n_init=20, random_state=seed).fit_predict(cluster_x))
        for seed in seeds
    ]
    sensitivity_rows.append({
        "k": k, "inertia": float(reference.inertia_),
        "silhouette": float(silhouette_score(cluster_x, reference.labels_)),
        "ari_seed_mean": float(np.mean(seed_aris)), "ari_seed_min": float(np.min(seed_aris)),
        "seeds": json.dumps(seeds),
    })
cluster_sensitivity = pd.DataFrame(sensitivity_rows)
zone_codes = pd.Categorical(cluster_pool.delivery_zone).codes
cluster_zone_ari = adjusted_rand_score(cluster_pool.cluster_id, zone_codes)
cluster_zone_cross = pd.crosstab(
    cluster_pool.geometric_name, cluster_pool.delivery_zone, margins=True
).reset_index()
assert len(cluster_pool) == 1493 and len(cluster_centers) == 4
assert cluster_pool.match_date.max() < EDA_END
display(cluster_centers.round(2))
display(cluster_sensitivity.round(3))
display(cluster_zone_cross)
print(f"ARI k=4 contra delivery_zone vigente: {cluster_zone_ari:.3f}")

design_scope_audit = pd.DataFrame([
    {"analysis": "manual threshold review", "max_date": review.match_date.max(),
     "strictly_before_eda_end": bool(review.match_date.max() < EDA_END), "uses_post_cutoff": False},
    {"analysis": "K-Means destinations", "max_date": cluster_pool.match_date.max(),
     "strictly_before_eda_end": bool(cluster_pool.match_date.max() < EDA_END), "uses_post_cutoff": False},
    {"analysis": "contract/count quality checks", "max_date": corners.match_date.max(),
     "strictly_before_eda_end": False, "uses_post_cutoff": True},
    {"analysis": "feature materialization (not EDA)", "max_date": corners.match_date.max(),
     "strictly_before_eda_end": False, "uses_post_cutoff": True},
])
assert design_scope_audit.loc[:1, "strictly_before_eda_end"].all()
display(design_scope_audit)

## 6. Contrato de salida

In [ ]:
feature_dictionary = pd.DataFrame([
    {"variable": name, "role": "pre_match_candidate", "availability": "before_match",
     "window": "eight_strictly_prior_matches", "meaning": meaning}
    for name, meaning in {
        "hist_corners_per_match": "Average attacking corners",
        "hist_scr15_smoothed": "Smoothed attacking SCR-15",
        "hist_xg_per_corner": "Historical post-corner xG, descriptive lag",
        "hist_short_share": "Historical short proxy share",
        "hist_high_share": "Historical high-pass share",
        "hist_top_taker_share": "Historical leading taker concentration",
        "hist_dominant_zone_share": "Historical dominant direct-delivery zone share",
        "hist_score_losing_share": "Historical share awarded while losing",
        "opp_hist_scr15_conceded_smoothed": "Opponent smoothed SCR-15 conceded",
        "is_home": "Target match venue flag",
    }.items()
])
decision_log = pd.DataFrame([
    {"decision": "source", "value": SOURCE_CONTRACT, "reason": "Audited SCR-15 input"},
    {"decision": "history", "value": "8", "reason": "Strict prior-match window"},
    {"decision": "short_proxy", "value": "18 units + Ground/Low", "reason": "Versioned descriptive proxy"},
    {"decision": "manual_labels", "value": label_audit["status"], "reason": "Integrity yes; independence unproven"},
    {"decision": "kmeans", "value": "k=4 development-only", "reason": "Descriptive, not predictive"},
])

exports = {
    "corners_engineered.parquet": corners,
    "team_match_observed.parquet": observed,
    "pre_match_features.parquet": features,
    "training_candidates_pre_match.parquet": training_pre,
    "training_candidates_scenario.parquet": training_scenario,
    "short_manual_review.parquet": review,
    "short_label_metrics.parquet": label_metrics,
    "short_threshold_sensitivity.parquet": threshold_sensitivity,
    "short_near_threshold.parquet": near_threshold,
    "feature_dictionary.parquet": feature_dictionary,
    "decision_log.parquet": decision_log,
    "history_perturbation_audit.parquet": perturbation_audit,
    "cluster_assignments.parquet": cluster_assignments,
    "cluster_centers.parquet": cluster_centers,
    "cluster_sensitivity.parquet": cluster_sensitivity,
    "cluster_zone_cross.parquet": cluster_zone_cross,
    "design_scope_audit.parquet": design_scope_audit,
    "model_scr15_pre_match.parquet": training_pre,
    "model_scr15_scenario.parquet": training_scenario,
    "model_short_direct.parquet": model_short_direct,
    "model_delivery_zone.parquet": model_delivery_zone,
    "model_corner_count.parquet": model_corner_count,
    "modeling_table_contract.parquet": modeling_table_contract,
}
export_records = []
for filename, frame in exports.items():
    path = OUT / filename
    frame.to_parquet(path, index=False)
    export_records.append({"file": filename, "rows": len(frame), "columns": len(frame.columns),
                           "sha256": digest(path)})
pd.testing.assert_frame_equal(
    pd.read_parquet(OUT / "short_manual_review.parquet")[compare_columns]
      .astype({"event_id": "string"}).sort_values("event_id").reset_index(drop=True),
    labels[compare_columns].sort_values("event_id").reset_index(drop=True),
    check_dtype=False,
)
write_json(OUT / "short_label_audit.json", label_audit)
export_records.append({"file": "short_label_audit.json", "rows": 1, "columns": len(label_audit),
                       "sha256": digest(OUT / "short_label_audit.json")})

contract04 = {
    "stage": "04_features", "contract_version": STAGE_VERSION, "run_id": RUN_ID,
    "source_contract_version": SOURCE_CONTRACT, "source_rule_version": RULE_VERSION,
    "source_run_id": contract03["run_id"], "source_corners_sha256": corner_export["sha256"],
    "parameters": {"history_matches": HISTORY_MATCHES, "short_threshold": SHORT_THRESHOLD,
                    "smoothing_strength": SMOOTHING_STRENGTH, "eda_end_exclusive": str(EDA_END.date()),
                    "league_prior": "expanding, valid corners with match_date strictly before target"},
    "counts": {"matches": 380, "corners": 3841, "evaluable": 3835, "excluded": 6,
               "team_matches": 760, "pre_match_ready": 600, "training_rows": 3039,
               "training_positive": 976, "manual_labels": 40, "cluster_rows": 1493,
               "valid_geometry": 3838, "direct_deliveries": 3374,
               "zero_corner_team_matches": 17},
    "core_features": CORE_FEATURES, "label_audit_status": label_audit["status"],
    "post_eda_period_status": "eligible for temporal evaluation but not a pristine untouched holdout: full-season contract and target counts were quality-checked; no post-cutoff EDA, threshold selection, or K-Means fitting",
    "clusters_allowed_as_model_features": False,
    "exports": export_records,
    "environment": {"python": platform.python_version(), "pandas": pd.__version__,
                    "numpy": np.__version__, "scikit_learn": sklearn.__version__},
}
write_json(OUT / "contract.json", contract04)
print(json.dumps(contract04["counts"], indent=2))

## Conclusion

La etapa entrega 3,039 corners evaluables con predictores construidos solo con
ocho partidos anteriores. Las 40 etiquetas heredadas son reproducibles pero no
constituyen validacion independiente; sirven para calibracion descriptiva del
proxy y deben reanotarse con una rubrica ciega. Ningun analisis de umbral,
EDA o K-Means usa fechas posteriores a `EDA_END`. La materializacion y los
controles de calidad si recorren la temporada completa, por lo que el periodo
posterior puede usarse para evaluacion temporal, pero no llamarse holdout
completamente virgen.